## Tools

In [1]:
import os
from langchain.chat_models import init_chat_model


model = init_chat_model("google_genai:gemini-3.8-flash")
response = model.invoke("Why do parrots talk?")
response

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


AIMessage(content=[{'type': 'text', 'text': 'Parrots talk—or more accurately, **mimic human speech**—because of a unique combination of extreme social intelligence, specialized brain anatomy, and physical vocal capabilities. \n\nHere is a breakdown of why and how they do it:\n\n---\n\n### 1. The Need to Fit In (The "Flock" Mentality)\nIn the wild, parrots are highly social creatures that live in tight-knit flocks. \n* **Contact Calls:** Wild parrots use distinct vocalizations to communicate, locate each other, and warn of danger. Each flock often develops its own "dialect." Young parrots learn these sounds to fit in and show they belong to the group.\n* **Humans as the New Flock:** When a parrot is kept as a pet, it views its human family as its flock. Because it has a deep, instinctual drive to bond and communicate, it begins to imitate the sounds its "flock" makes—which happen to be human words.\n\n### 2. A Brain Wired for Mimicry\nParrots are part of a rare group of animals known as

In [10]:
from langchain.tools import tool

@tool
def get_weather(location: str) -> str:
    """Get the weather in a given location"""
    return f"Its sunny in {location}"

model_with_tools= model.bind_tools([get_weather])

In [14]:
response = model_with_tools.invoke("What is the weather in New York?")
print(response)

for tool_call in response.tool_calls:
    print(f"Tool: {tool_call['name']}")
    print(f"Arguments: {tool_call['args']}")

content=[] additional_kwargs={'function_call': {'name': 'get_weather', 'arguments': '{"location": "New York"}'}, '__gemini_function_call_thought_signatures__': {'call_439074': 'EqICCp8CAWkUfRNcy/f9zxr3RB13o37125td46rL76bz7A61rXNSzot0xgXXckYb9pTH+iM0BDBhMXaJUKAZwXpaAN0lGTPvcbJQWyb1IHMxLBwR/+0N7enYJLvTxb5Ya21Fm8RpDuncvAQB6KiInUqczTFUnxZ+kcC9OAAdFUE7cZO/GYOZoUbk62VQvW2gu8XhOBeooNil3UlF+jJ6nN9wAoudSza0ky7bghEXPavK7hewa0sLF6bqoVN8u6qzV+OLvK2pvxhZO3gZ/CvYk5MEnu3ldJ2BJISuj4jYh0YRYkYTdDmU/UQh7hJRvA/c8KOIX/ywVznERYGHRqGj0d9m/8evrqJPsFlgjAF5gjUWgEc1MicbCm0rAsJBV+vFE9F9kD4='}} response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.8-flash', 'safety_ratings': [], 'model_provider': 'google_genai'} id='lc_run--01a0e484-c6e1-7af1-b315-6732a3b33a45-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'New York'}, 'id': 'call_439074', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 53, 'output_tokens': 65, 'total_tokens': 118, 'input_token_details': {'cac

### Tool Execution Loops

In [15]:
# Step 1: Model generates tool calls
messages = [{"role": "user", "content": "What's the weather in New York?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# Step 3: Pass results back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)

The weather in New York is currently sunny.


In [17]:
messages

[{'role': 'user', 'content': "What's the weather in New York?"},
 AIMessage(content=[], additional_kwargs={'function_call': {'name': 'get_weather', 'arguments': '{"location": "New York"}'}, '__gemini_function_call_thought_signatures__': {'call_312690': 'Eo4CCosCAWkUfRN26tN0gok5NB3CO31/yls0D1rAouspYa6qcr6JTaeljviUfk60FUEhCUzFQBW/AdRunbCOWqfNv41hGy3qc3RMLTJbIRNKjZ55NLtOXCze36aasEu6WZSXThpP9EG7YrVROZOFK9JhBf7+ZWB261O8SILUQWEtEKG6Jt3YhncYP8y1KTjNeNivEguVVTcVtCgUri7HFTXI1X0Y1AUZpe6rmD3+xql6DwB3R5DrmAOFA8BNO68soshvZA3pRf7osnDjjLe2R1adP9AW0s/MA5K9Ntv6prqjhLzBhGO/L6X16I0MCB/2P2E7+u4N2pRd/OcT3Kk1tTxFIDSBzIbA8DxGE5kBTCex'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.8-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a0e484-d9d9-71d3-aaa6-0463b8dc9fcc-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'New York'}, 'id': 'call_312690', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 54, 'output_tokens':